In [ ]:
# Safe imports and optimizer fallbacks — run this cell first to avoid tensorflow_addons import errors
import sys, os
import numpy as np, pandas as pd
import tensorflow as tf

TFA_AVAILABLE = False
try:
    import tensorflow_addons as tfa
    TFA_AVAILABLE = True
    AdamW = tfa.optimizers.AdamW
    print('tensorflow_addons available — using tfa.optimizers.AdamW')
except Exception as e:
    print('tensorflow_addons not available:', e)
    # Try keras.experimental optimizer or tf.keras implementation
    try:
        # TF 2.11+ provides AdamW in tf.keras.optimizers
        from tensorflow.keras.optimizers import AdamW
        print('Using tensorflow.keras.optimizers.AdamW')
    except Exception:
        try:
            # Some TF versions expose AdamW in experimental namespace
            from tensorflow.keras.optimizers import adamw_v2 as AdamW_impl
            AdamW = AdamW_impl.AdamW
            print('Using experimental AdamW implementation')
        except Exception:
            # Fallback to plain Adam (no weight decay). Weight decay can be emulated via kernel_regularizer.
            from tensorflow.keras.optimizers import Adam
            AdamW = Adam
            print('Falling back to Adam (no built-in weight decay)')

print('TensorFlow version:', tf.__version__, 'TFA_AVAILABLE=', TFA_AVAILABLE)


def get_optimizer(lr=1e-4, weight_decay=1e-4):
    """Return an optimizer (AdamW if available, else Adam)."""
    if TFA_AVAILABLE:
        return tfa.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    else:
        try:
            # keras AdamW signature may accept weight_decay
            return AdamW(learning_rate=lr, weight_decay=weight_decay)
        except TypeError:
            # Fall back
            return tf.keras.optimizers.Adam(learning_rate=lr)



In [ ]:
# Minimal variables for use by later cells
PROJECT_NAME = "CDD multimodal CV"
GOALS = [
    "Robust data loading and alignment",
    "TF dataset and augmentation",
    "Stratified k-fold training",
    "Class-balanced focal loss + AdamW",
    "Ensemble inference & diagnostics"
]
EXPECTED_OUTPUT = {
    'per_fold_metrics_csv': 'model/metrics_cv.csv',
    'ensemble_model_dir': 'model/ensemble/',
}
print('PROJECT:', PROJECT_NAME)

In [ ]:
# Install optional packages (guarded) — do NOT force-install tensorflow_addons here; just detect availability
import sys
import importlib

# helper to install packages if the user explicitly requests it (kept but not used automatically)
def pip_install(pkg):
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# Check for tensorflow_addons without installing it automatically (avoid environment issues)
TFA_PRESENT = False
try:
    tfa_spec = importlib.util.find_spec('tensorflow_addons')
    if tfa_spec is not None:
        import tensorflow_addons as tfa
        TFA_PRESENT = True
        print('tensorflow_addons detected:', getattr(tfa, '__version__', 'unknown'))
    else:
        print('tensorflow_addons not detected; continuing without it (get_optimizer fallback will be used)')
except Exception as e:
    print('Error checking tensorflow_addons:', e)

# Import and print versions (safe)
import os, platform
import numpy as np, pandas as pd
import tensorflow as tf
import sklearn

print('Python:', sys.version)
print('TF:', tf.__version__)
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('Scikit-learn:', sklearn.__version__)

# Set seed for reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)

In [ ]:
# Configuration
CONFIG = {
    'image_dir_le': 'CDD-CESM/PKG - CDD-CESM/CDD-CESM/Low energy images of CDD-CESM',
    'image_dir_sub': 'CDD-CESM/PKG - CDD-CESM/CDD-CESM/Subtracted images of CDD-CESM',
    'json_dir': 'CDD-CESM/json_output',
    'excel_path': 'processed_metadata.csv',
    'img_size': (224,224),
    'n_views': 8,
    'tfidf_max_features': 1000,
    'batch_size': 8,
    'epochs': 20,
    'k_folds': 5,
    'seed': SEED,
    'radimagenet_weights': 'weights/RadImageNet-DenseNet121_notop.h5',
    'save_features_dir': 'model',
    'model_dir': 'model/cv_folds',
    'ensemble_dir': 'model/ensemble',
}

os.makedirs(CONFIG['save_features_dir'], exist_ok=True)
os.makedirs(CONFIG['model_dir'], exist_ok=True)
os.makedirs(CONFIG['ensemble_dir'], exist_ok=True)

# Quick checks
for p in ['image_dir_le', 'image_dir_sub', 'json_dir', 'excel_path']:
    print(p, '->', CONFIG[p], 'exists=', os.path.exists(CONFIG[p]))

In [ ]:
from PIL import Image
import numpy as np
import pandas as pd
import json
import os


def load_metadata(excel_path):
    df = pd.read_csv(excel_path)
    df = df.dropna(subset=['Patient_ID', 'Pathology Classification/ Follow up'])
    df['Patient_ID'] = df['Patient_ID'].astype(str)
    return df


def load_texts_for_df(df, json_dir):
    texts = {}
    for pid in df['Patient_ID']:
        path = os.path.join(json_dir, f"P{pid}.json")
        if os.path.exists(path):
            try:
                with open(path, encoding='utf-8') as f:
                    d = json.load(f)
                flat = []
                for v in d.values():
                    flat.extend(map(str, v) if isinstance(v, list) else [str(v)])
                texts[str(pid)] = ' '.join(flat)
            except Exception:
                texts[str(pid)] = ''
        else:
            texts[str(pid)] = ''
    return texts


def canonical_patient_ids(df, sampled_ids=None):
    # Return sorted canonical IDs to ensure deterministic ordering
    if sampled_ids is None:
        pids = sorted(df['Patient_ID'].astype(str).unique())
    else:
        pids = sorted(list(map(str, sampled_ids)))
    return pids


def load_images_sequence(pid, image_dir_le, image_dir_sub, img_size=(224,224), n_views=8):
    H,W = img_size
    paths = [
        f"{image_dir_le}/P{pid}_L_DM_CC.jpg",
        f"{image_dir_le}/P{pid}_L_DM_MLO.jpg",
        f"{image_dir_sub}/P{pid}_L_CM_CC.jpg",
        f"{image_dir_sub}/P{pid}_L_CM_MLO.jpg",
        f"{image_dir_le}/P{pid}_R_DM_CC.jpg",
        f"{image_dir_le}/P{pid}_R_DM_MLO.jpg",
        f"{image_dir_sub}/P{pid}_R_CM_CC.jpg",
        f"{image_dir_sub}/P{pid}_R_CM_MLO.jpg",
    ][:n_views]
    imgs = []
    found_any = False
    for p in paths:
        if os.path.exists(p):
            try:
                im = Image.open(p).convert('L').resize((W,H))
                arr = np.array(im, dtype=np.float32)/255.0
                arr = arr[..., np.newaxis]
                imgs.append(arr)
                found_any = True
            except Exception:
                imgs.append(np.zeros((H,W,1), dtype=np.float32))
        else:
            imgs.append(np.zeros((H,W,1), dtype=np.float32))
    return np.stack(imgs, axis=0) if found_any else None

print('Utilities defined')

In [ ]:


from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight


def prepare_arrays(cfg=CONFIG, sample_frac=0.8):
    df = load_metadata(cfg['excel_path'])
    sampled_df = df.sample(frac=sample_frac, random_state=cfg['seed'])
    sampled_pids = sampled_df['Patient_ID'].astype(str).tolist()

    texts_map = load_texts_for_df(sampled_df, cfg['json_dir'])

    images = []
    labels = []
    pids = []
    for pid, label in zip(sampled_df['Patient_ID'].astype(str), sampled_df['Pathology Classification/ Follow up']):
        seq = load_images_sequence(pid, cfg['image_dir_le'], cfg['image_dir_sub'], cfg['img_size'], cfg['n_views'])
        if seq is not None:
            images.append(seq)
            labels.append(label)
            pids.append(pid)

    # Canonical ordering
    common = sorted(list(set(pids) & set(sampled_pids)))
    images_filtered = np.array([ {pid:img for pid,img in zip(pids,images)}[pid] for pid in common ])
    labels_filtered = [ {pid:lab for pid,lab in zip(pids,labels)}[pid] for pid in common ]
    texts_filtered = [ texts_map[pid] for pid in common ]

    meta_df_filtered = pd.DataFrame([{ 'Patient_ID':pid, **(load_metadata(cfg['excel_path'])[load_metadata(cfg['excel_path'])['Patient_ID']==pid].iloc[0].to_dict() if True else {}) } for pid in common])

    # TF-IDF
    vect = TfidfVectorizer(max_features=cfg['tfidf_max_features'])
    text_feats = vect.fit_transform(texts_filtered).toarray()

    # Meta
    df_meta_full = load_metadata(cfg['excel_path'])
    meta_rows = [df_meta_full[df_meta_full['Patient_ID']==pid].iloc[0] for pid in common]
    meta_df = pd.DataFrame(meta_rows)

    numerical = meta_df.select_dtypes(include=['float','int']).columns.tolist()
    categorical = meta_df.select_dtypes(include=['object']).drop(columns=['Patient_ID','Pathology Classification/ Follow up']).columns.tolist()

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    meta_num = scaler.fit_transform(meta_df[numerical]) if numerical else np.zeros((len(meta_df),0))
    meta_cat = encoder.fit_transform(meta_df[categorical]) if categorical else np.zeros((len(meta_df),0))
    meta_feats = np.concatenate([meta_num, meta_cat], axis=1) if (meta_num.size or meta_cat.size) else np.zeros((len(meta_df),0))

    # Labels
    labels_enc, label_names = pd.factorize(labels_filtered)
    labels_cat = tf.keras.utils.to_categorical(labels_enc)

    print('Prepared arrays: N=', images_filtered.shape[0], 'labels=', np.unique(labels_enc, return_counts=True))
    return images_filtered, meta_feats, text_feats, labels_cat, labels_enc, label_names, vect, scaler, encoder

# Run preparation (light)
images_filtered, meta_feats, text_feats, labels_cat, labels_enc, label_names, vect, scaler, encoder = prepare_arrays(CONFIG)

print('Shapes:', images_filtered.shape, meta_feats.shape, text_feats.shape, labels_cat.shape)



In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# Augmentation — per-view applied but deterministic within sequence through stateless ops
def get_augmentation(img_size=(224,224)):
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.05),
        # brightness/contrast as lambda
        layers.Lambda(lambda x: tf.image.random_brightness(x, 0.05)),
        layers.Lambda(lambda x: tf.image.random_contrast(x, 0.05, 0.15)),
    ], name='aug')

aug_layer = get_augmentation(CONFIG['img_size'])

# Mixup helper
def mixup(batch_x1, batch_x2, batch_y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    x = lam * batch_x1 + (1 - lam) * batch_x2
    y = lam * batch_y + (1 - lam) * batch_y[::-1]
    return x, y

# TF dataset builder

def build_tf_dataset(X_img, X_meta, X_txt, y, batch_size=8, augment=False, shuffle=True):
    """Returns a tf.data.Dataset yielding ((img_seq, meta, txt), label) batches."""
    ds = tf.data.Dataset.from_tensor_slices((X_img.astype('float32'), X_meta.astype('float32'), X_txt.astype('float32'), y.astype('float32')))
    if shuffle:
        ds = ds.shuffle(buffer_size=1024, seed=CONFIG['seed'])
    def _map(img_seq, meta, txt, lab):
        # img_seq: (n_views,H,W,1)
        img_seq = tf.cast(img_seq, tf.float32)
        meta = tf.cast(meta, tf.float32)
        txt = tf.cast(txt, tf.float32)
        lab = tf.cast(lab, tf.float32)
        if augment:
            # flatten views, augment, then restore
            n_views = tf.shape(img_seq)[0]
            H = tf.shape(img_seq)[1]; W = tf.shape(img_seq)[2]; C = tf.shape(img_seq)[3]
            flat = tf.reshape(img_seq, (n_views, H, W, C))
            aug = aug_layer(flat, training=True)
            img_seq = tf.reshape(aug, (n_views, H, W, C))
        # return inputs as a tuple and label separately to match Keras expected (x, y)
        return (img_seq, meta, txt), lab
    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

print('TF dataset builder ready')

In [ ]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.resnet50 import preprocess_input


def create_backbone(img_size=(224,224), output_dim=64, weight_path=None, fine_tune_at=None):
    try:
        base = DenseNet121(include_top=False, weights=None, input_shape=(img_size[0], img_size[1], 3))
        if weight_path and os.path.exists(weight_path):
            base.load_weights(weight_path)
            print('Loaded RadImageNet weights')
        else:
            # fallback to imagenet weights
            base = DenseNet121(include_top=False, weights='imagenet', input_shape=(img_size[0], img_size[1], 3))
            print('Using ImageNet weights')
        if fine_tune_at is None:
            base.trainable = False
        else:
            base.trainable = True
            for layer in base.layers[:fine_tune_at]:
                layer.trainable = False
        inp = layers.Input(shape=(img_size[0], img_size[1], 1))
        x = layers.Lambda(lambda t: tf.image.grayscale_to_rgb(t))(inp)
        x = layers.Lambda(lambda z: preprocess_input(z))(x)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(output_dim, activation='relu')(x)
        return tf.keras.Model(inp, x, name='backbone')
    except Exception as e:
        print('Backbone creation failed:', e)
        # fallback small cnn
        inp = layers.Input(shape=(img_size[0], img_size[1], 1))
        x = layers.Conv2D(32,3,activation='relu',padding='same')(inp)
        x = layers.MaxPool2D(2)(x)
        x = layers.Conv2D(64,3,activation='relu',padding='same')(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(output_dim,activation='relu')(x)
        return tf.keras.Model(inp,x,name='small_cnn_backbone')


def build_multimodal_model(img_feat_dim=64, meta_dim=0, text_dim=1000, n_classes=3, cfg=CONFIG, l2_reg=1e-4, dropout_rate=0.3):
    # inputs
    img_feats_input = layers.Input(shape=(cfg['n_views'], cfg['img_size'][0], cfg['img_size'][1],1), name='img_seq')
    meta_in = layers.Input(shape=(meta_dim,), name='meta')
    text_in = layers.Input(shape=(text_dim,), name='text')

    # TimeDistributed backbone
    backbone = create_backbone(img_size=cfg['img_size'], output_dim=img_feat_dim, weight_path=cfg['radimagenet_weights'], fine_tune_at=None)
    td = layers.TimeDistributed(backbone)(img_feats_input)  # (batch, views, feat)

    # Transformer block
    x = TransformerBlock(embed_dim=img_feat_dim, num_heads=4, ff_dim=128)(td)
    x = layers.GlobalAveragePooling1D()(x)

    # meta/text branches
    m = layers.Dense(64, activation='relu', kernel_regularizer=l2(l2_reg))(meta_in)
    t = layers.Dense(64, activation='relu', kernel_regularizer=l2(l2_reg))(text_in)

    x = layers.Concatenate()([x, m, t])
    x = layers.Dense(128, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate*0.75)(x)
    out = layers.Dense(n_classes, activation='softmax', name='out')(x)

    model = tf.keras.Model(inputs=[img_feats_input, meta_in, text_in], outputs=out)
    return model

print('Model builder ready')

In [ ]:
# TransformerBlock definition (required by model builder)
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim),
        ])
        self.ln1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.do1 = Dropout(rate)
        self.do2 = Dropout(rate)

    def call(self, inputs, training=None):
        attn = self.att(inputs, inputs)
        attn = self.do1(attn, training=training)
        out1 = self.ln1(inputs + attn)
        ffn = self.ffn(out1)
        ffn = self.do2(ffn, training=training)
        return self.ln2(out1 + ffn)

print('TransformerBlock defined')

In [ ]:
from tensorflow.keras import backend as K

# focal loss
def focal_loss(gamma=2.0, alpha=None):
    def loss(y_true, y_pred):
        y_pred = K.clip(y_pred, K.epsilon(), 1. - K.epsilon())
        cross_entropy = -y_true * K.log(y_pred)
        if alpha is not None:
            alpha_factor = y_true * alpha
        else:
            alpha_factor = 1.0
        weight = alpha_factor * K.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return K.mean(K.sum(loss, axis=-1))
    return loss

# compile helper — use get_optimizer() defined in the safe imports cell which provides AdamW if available or falls back to Adam

def compile_model(model, lr=1e-4, weight_decay=1e-4, gamma=2.0, alpha=None):
    opt = get_optimizer(lr=lr, weight_decay=weight_decay)
    model.compile(optimizer=opt, loss=focal_loss(gamma=gamma, alpha=alpha), metrics=['accuracy'])
    return model

print('Loss & optimizer helpers ready')


## Section 6 — Training loop: Stratified K-Fold CV

Train with k-fold and save best model per fold. Aggregate metrics.

In [ ]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, roc_auc_score

k = CONFIG['k_folds']
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=CONFIG['seed'])

metrics_per_fold = []
models = []

X_img = images_filtered
X_meta = meta_feats
X_txt = text_feats
y_cat = labels_cat
y_idx = labels_enc

fold = 0
for train_idx, val_idx in skf.split(X_img, y_idx):
    fold += 1
    print('\n=== Fold', fold, '===')
    X_img_train, X_img_val = X_img[train_idx], X_img[val_idx]
    X_meta_train, X_meta_val = X_meta[train_idx], X_meta[val_idx]
    X_txt_train, X_txt_val = X_txt[train_idx], X_txt[val_idx]
    y_train, y_val = y_cat[train_idx], y_cat[val_idx]

    # Oversampling optional: simple repeat minority in training set (fast path)
    from collections import Counter
    counts = Counter(np.argmax(y_train, axis=1))
    max_c = max(counts.values())
    X_img_train_os, X_meta_train_os, X_txt_train_os, y_train_os = [],[],[],[]
    for cls in range(y_train.shape[1]):
        idxs = np.where(np.argmax(y_train, axis=1)==cls)[0]
        if len(idxs)==0: continue
        reps = int(np.ceil(max_c/len(idxs)))
        sel = np.tile(idxs, reps)[:max_c]
        for i in sel:
            X_img_train_os.append(X_img_train[i])
            X_meta_train_os.append(X_meta_train[i])
            X_txt_train_os.append(X_txt_train[i])
            y_train_os.append(y_train[i])
    X_img_train_os = np.array(X_img_train_os)
    X_meta_train_os = np.array(X_meta_train_os)
    X_txt_train_os = np.array(X_txt_train_os)
    y_train_os = np.array(y_train_os)

    print('Train shapes after OS:', X_img_train_os.shape, X_meta_train_os.shape, X_txt_train_os.shape, y_train_os.shape)

    # Build tf datasets
    train_ds = build_tf_dataset(X_img_train_os, X_meta_train_os, X_txt_train_os, y_train_os, batch_size=CONFIG['batch_size'], augment=True, shuffle=True)
    val_ds = build_tf_dataset(X_img_val, X_meta_val, X_txt_val, y_val, batch_size=CONFIG['batch_size'], augment=False, shuffle=False)

    # Build model
    model = build_multimodal_model(img_feat_dim=64, meta_dim=X_meta.shape[1], text_dim=X_txt.shape[1], n_classes=y_cat.shape[1])
    model = compile_model(model, lr=1e-4, weight_decay=1e-4, gamma=2.0, alpha=None)
    model.summary()

    # Callbacks
    cb = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(os.path.join(CONFIG['model_dir'], f'best_fold_{fold}.h5'), monitor='val_loss', save_best_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
    ]

    hist = model.fit(train_ds, validation_data=val_ds, epochs=CONFIG['epochs'], callbacks=cb)

    # Save history
    try:
        np.save(os.path.join(CONFIG['model_dir'], f'history_fold_{fold}.npy'), hist.history)
    except Exception:
        pass

    # Evaluate on val
    preds = model.predict(val_ds)
    y_pred = np.argmax(preds, axis=1)
    y_true = np.argmax(y_val, axis=1)

    report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True)
    print('Fold', fold, 'report:\n', classification_report(y_true, y_pred, target_names=label_names))
    cm = confusion_matrix(y_true, y_pred)
    print('Confusion matrix:\n', cm)

    # save model and metrics
    model.save(os.path.join(CONFIG['model_dir'], f'model_fold_{fold}.h5'))
    metrics_per_fold.append({'fold':fold, 'report':report, 'cm':cm})
    models.append(model)

# Save aggregate metrics
import csv
with open(os.path.join(CONFIG['model_dir'], 'metrics_cv.csv'), 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['fold','class','precision','recall','f1','support'])
    for m in metrics_per_fold:
        for cls in label_names:
            d = m['report'][cls]
            writer.writerow([m['fold'], cls, d['precision'], d['recall'], d['f1-score'], d['support']])

print('CV training complete')